In [11]:
%pip install -qU langchain langchain-ollama langgraph pydantic

Note: you may need to restart the kernel to use updated packages.


In [12]:
import subprocess
import time

print("Iniciando o servidor Ollama...")
process = subprocess.Popen(["ollama", "serve"])
time.sleep(3) 
# Faz o download do modelo Llama 3.1 
print("Baixando o modelo Llama 3.1 (8B)...")
!ollama pull llama3.1
print("Modelo pronto para uso.")

Iniciando o servidor Ollama...


FileNotFoundError: [WinError 2] O sistema não pode encontrar o arquivo especificado

In [ ]:
from langchain_core.tools import tool

# --- Simuladores RAG ---
def retrieve_triage_protocols(symptoms: str) -> str:
    protocols = {
        "dor no peito": "PROTOCOLO: Dor no peito irradiando para o braço requer escalonamento de emergência imediato. Não prossiga com a triagem padrão.",
        "dor de cabeça": "PROTOCOLO: Avalie febre, rigidez na nuca e fotofobia. Se ausentes, classifique como baixo risco."
    }
    for key, protocol in protocols.items():
        if key in symptoms.lower():
            return protocol
    return "PROTOCOLO: Avaliação padrão. Colete o início e a gravidade dos sintomas."

def retrieve_patient_history(patient_id: str) -> str:
    db = {
        "ID-123": "Alergias: Penicilina. Crônico: Hipertensão. Medicações Atuais: Losartana 50mg."
    }
    return db.get(patient_id, "Nenhum histórico encontrado.")

# --- Ferramentas: Pilar de Triagem ---
@tool
def get_wearable_data(patient_id: str) -> str:
    """Extrai batimentos cardíacos e SpO2 do Apple Health/Google Fit."""
    return f"Dados para {patient_id}: Frequência Cardíaca 85 bpm, SpO2 98%."

@tool
def register_symptoms(patient_id: str, symptoms_list: list) -> str:
    """Estrutura e salva os sintomas do paciente no banco de dados."""
    return f"Sintomas {symptoms_list} registrados com sucesso para {patient_id}."

@tool
def trigger_red_flag(reason: str) -> str:
    """Aciona o protocolo de emergência. Chame imediatamente se sintomas críticos forem detectados."""
    return "ALERTA VERMELHO ACIONADO"

# --- Ferramentas: Pilar Copiloto ---
@tool
def check_drug_interaction(new_drug: str, current_drugs: list) -> str:
    """Valida se o novo medicamento interage negativamente com as medicações atuais."""
    if "Ibuprofeno" in new_drug and "Losartana" in current_drugs:
        return "AVISO: O Ibuprofeno pode reduzir o efeito anti-hipertensivo da Losartana."
    return "Nenhuma interação grave detectada."

@tool
def draft_prescription(medications: list, dosage: str) -> str:
    """Formata a prescrição para aprovação médica."""
    return f"RASCUNHO DE PRESCRIÇÃO PRONTO. Medicamentos: {medications}, Dosagem: {dosage}. Aguardando assinatura."

triage_tools = [get_wearable_data, register_symptoms, trigger_red_flag]
copilot_tools = [check_drug_interaction, draft_prescription]

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    role: Literal["patient", "doctor"]
    patient_id: str

# Configuração do Llama 3.1 via Ollama
llm = ChatOllama(model="llama3.1", temperature=0)

triage_sys_prompt = """Você é o Agente de Triagem BluaDiagnostics.
1. Use o contexto fornecido para guiar suas perguntas.
2. Use as ferramentas para obter dados de wearables ou registrar sintomas.
3. CRÍTICO: Se detectar dor no peito, sinais de AVC ou falta de ar grave, você DEVE chamar a ferramenta `trigger_red_flag` imediatamente."""

copilot_sys_prompt = """Você é o Copiloto Clínico BluaDiagnostics.
1. Você auxilia médicos após a teleconsulta.
2. Revise o histórico do paciente.
3. Sempre chame `check_drug_interaction` antes de criar o rascunho.
4. Chame `draft_prescription` para finalizar. Não prescreva diretamente aos pacientes."""

# --- Definições dos Nós ---
def triage_node(state: AgentState):
    user_msg = state["messages"][-1].content
    protocol_context = retrieve_triage_protocols(user_msg)
    
    messages = [
        SystemMessage(content=f"{triage_sys_prompt}\n\nContexto: {protocol_context}")
    ] + state["messages"]
    
    # Vincula as ferramentas ao Llama 3.1
    llm_triage = llm.bind_tools(triage_tools)
    response = llm_triage.invoke(messages)
    return {"messages": [response]}

def emergency_node(state: AgentState):
    escalation_msg = "Protocolo de Emergência Ativado: Redirecionando para teleconsulta de urgência imediatamente."
    return {"messages": [SystemMessage(content=escalation_msg)]}

def copilot_node(state: AgentState):
    history_context = retrieve_patient_history(state.get("patient_id", ""))
    messages = [
        SystemMessage(content=f"{copilot_sys_prompt}\n\nHistórico do Paciente: {history_context}")
    ] + state["messages"]
    
    # Vincula as ferramentas ao Llama 3.1
    llm_copilot = llm.bind_tools(copilot_tools)
    response = llm_copilot.invoke(messages)
    return {"messages": [response]}

# --- Lógica de Rotas Condicionais ---
def check_red_flag(state: AgentState) -> Literal["emergency_node", "tools", "__end__"]:
    last_message = state["messages"][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        for tool_call in last_message.tool_calls:
            if tool_call["name"] == "trigger_red_flag":
                return "emergency_node"
        return "tools"
    return "__end__"

def route_by_role(state: AgentState) -> Literal["triage_node", "copilot_node"]:
    if state["role"] == "patient":
        return "triage_node"
    return "copilot_node"

ValueError: At 'triage_node' node, 'check_red_flag' branch found unknown target 'tools'

In [ ]:
workflow = StateGraph(AgentState)

# Adicionar Nós
workflow.add_node("triage_node", triage_node)
workflow.add_node("emergency_node", emergency_node)
workflow.add_node("copilot_node", copilot_node)
workflow.add_node("triage_tools", ToolNode(triage_tools))
workflow.add_node("copilot_tools", ToolNode(copilot_tools))

# Adicionar Roteamento Inicial
workflow.add_conditional_edges(START, route_by_role)

# Fluxo de Triagem
workflow.add_conditional_edges("triage_node", check_red_flag)
workflow.add_edge("triage_tools", "triage_node")
workflow.add_edge("emergency_node", END)

# Fluxo do Copiloto
workflow.add_conditional_edges("copilot_node", tools_condition, {"tools": "copilot_tools", "__end__": END})
workflow.add_edge("copilot_tools", "copilot_node")

app = workflow.compile()
print("Grafo compilado e pronto para execução.")

In [ ]:
# Função para testar o fluxo
def testar_fluxo(mensagem, papel, patient_id="ID-123"):
    print(f"\n--- Testando fluxo ({papel}) ---")
    print(f"Entrada: {mensagem}")
    
    estado_inicial = {
        "messages": [("user", mensagem)],
        "role": papel,
        "patient_id": patient_id
    }
    
    for evento in app.stream(estado_inicial, stream_mode="values"):
        ultima_mensagem = evento["messages"][-1]
        if hasattr(ultima_mensagem, 'content') and ultima_mensagem.content:
             print(f"Agente: {ultima_mensagem.content}")
        elif hasattr(ultima_mensagem, 'tool_calls') and ultima_mensagem.tool_calls:
             print(f"Agente chamando ferramenta: {ultima_mensagem.tool_calls[0]['name']}")

# Teste 1: Triagem normal
testar_fluxo("Estou com uma dor de cabeça leve.", "patient")

# Teste 2: Triagem Red Flag (Aciona ferramenta trigger_red_flag)
testar_fluxo("Estou com dor no peito muito forte.", "patient")

# Teste 3: Copiloto Médico (Verifica interação de medicamentos)
testar_fluxo("Gere uma prescrição de Ibuprofeno 400mg.", "doctor")